# Chapter 8: Unsupervised Learning Techniques

While supervised learning is incredibly powerful, the vast majority of data available in the real world is **unlabeled** (we have the input features $X$, but no labels $y$). 

Yann LeCun famously compared AI to a cake: 
*   **Unsupervised Learning** is the cake itself (the foundation and bulk of the intelligence).
*   **Supervised Learning** is the icing on the cake.
*   **Reinforcement Learning** is the cherry on top.

**The Labeling Bottleneck:**
Consider a manufacturing production line. Taking thousands of pictures of items every day is trivial. However, manually labeling each picture as "defective" or "normal" to train a regular binary classifier is costly, slow, and often impractical. Unsupervised learning algorithms help us extract patterns and anomalies from these massive unlabeled datasets automatically.

## 1. Clustering: The Unsupervised Counterpart to Classification

In supervised learning (like **Classification**), we feed the algorithm with data that already has labels (e.g., we know exactly which flower is a *Setosa* or a *Virginica*). The goal is to predict the label of new instances.

In unsupervised learning (like **Clustering**), the dataset is completely unlabeled. The algorithm only sees input features (like an unorganized cloud of black data points). The goal of clustering is to automatically discover hidden structures and group similar instances together into clusters.

### A Sneak Peek: The Power of Clustering
To demonstrate how powerful this can be, we can test an unsupervised algorithm on a dataset where we *actually* know the labels (like the Iris dataset), but we hide those labels from the algorithm.

If we ask a **Gaussian Mixture Model** (a powerful clustering algorithm we will explore later) to group the unlabeled Iris data into 3 clusters, it successfully identifies the three distinct flower species based solely on their petal and sepal measurements!

```python
from scipy import stats
from sklearn.mixture import GaussianMixture
import numpy as np

# Fit the unsupervised model (it only sees X, not y!)
y_pred = GaussianMixture(n_components=3, random_state=42).fit(X).predict(X)

# Map the arbitrary cluster IDs (0, 1, 2) to the actual class labels to check accuracy
mapping = {}
for class_id in np.unique(y):
    mode, _ = stats.mode(y_pred[y==class_id])
    mapping[mode] = class_id

y_pred_mapped = np.array([mapping[cluster_id] for cluster_id in y_pred])

# Calculate the ratio of perfectly clustered instances
accuracy = (y_pred_mapped == y).sum() / len(y_pred_mapped)
print(f"Clustering Accuracy: {accuracy:.3f}")
# Output: Clustering Accuracy: 0.967
```
*Note: Achieving over 96% accuracy without ever showing the algorithm a single label proves the immense potential of unsupervised learning.*

## 2. K-Means Clustering

K-Means is a simple and elegant algorithm capable of clustering an unlabeled dataset quickly and efficiently. Geometrically, it looks for proximity (distance). It tries to group data points around central hubs, called **centroids**, ensuring that each point is assigned to the cluster whose centroid is closest to it.

### Training a K-Means Model
Let's generate a fake dataset consisting of 5 distinct blobs of data and train a K-Means model on it. We must specify the number of clusters (`n_clusters`) we expect the algorithm to find.

```python
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
import numpy as np

# 1. Generate a dataset of 5 blobs
blob_centers = np.array(
    [[ 0.2,  2.3], [-1.5 ,  2.3], [-2.8,  1.8], [-2.8,  2.8], [-2.8,  1.3]])
blob_std = np.array([0.4, 0.3, 0.1, 0.1, 0.1])
X, y = make_blobs(n_samples=2000, centers=blob_centers, cluster_std=blob_std, random_state=7)

# 2. Train the K-Means clusterer
k = 5
kmeans = KMeans(n_clusters=k, random_state=42)
y_pred = kmeans.fit_predict(X)
```

### Centroids and Predictions
Once trained, the algorithm has estimated the coordinates of the 5 centroids. We can view them using the `cluster_centers_` attribute.

```python
# View the estimated centroids
kmeans.cluster_centers_

# Predict the cluster for new instances
X_new = np.array([[0, 2], [3, 2], [-3, 3], [-3, 2.5]])
kmeans.predict(X_new)
# Output: array([1, 1, 2, 2], dtype=int32)
```

### Decision Boundaries (Voronoi Tessellation)
When you plot the decision boundaries of a K-Means algorithm, you get a **Voronoi diagram**. Each region in this diagram belongs to one centroid. Any new instance that falls into a specific polygon will be assigned to that polygon's central hub, simply because it is mathematically the closest one.

*Note: While K-Means performs very well on roughly spherical blobs, some instances near the edges of adjacent clusters might be misclassified because K-Means only cares about the distance to the centroids.*

### 3. Hard Clustering vs. Soft Clustering

When assigning instances to clusters in K-Means, you have two choices:

*   **Hard Clustering:** Assigning each instance strictly to a single cluster (the closest one). This is done using the `predict()` method, which outputs a single cluster index for each instance.
*   **Soft Clustering:** Instead of an arbitrary assignment, this approach gives each instance a score per cluster. In the case of K-Means, these scores are the **Euclidean distances** from the instance to each of the $k$ centroids. This is done using the `transform()` method.

```python
# 1. Hard Clustering (predicts the exact cluster ID)
kmeans.predict(X_new)
# Output: array([1, 1, 2, 2], dtype=int32)

# 2. Soft Clustering (measures the distance to all 5 centroids)
kmeans.transform(X_new).round(2)
# Output example for the first instance: [2.81, 0.33, 2.9, 1.49, 2.89]
# Notice that the shortest distance (0.33) corresponds to cluster index 1!
```

**Why use Soft Clustering?** 
If an instance is located near the decision boundary between two clusters, hard clustering forces an arbitrary choice. Soft clustering preserves the nuance (e.g., showing that the instance is almost equally close to two centroids). These distances can be incredibly useful as new features for a downstream supervised learning model!

### 4. The K-Means Algorithm Under the Hood

The K-Means algorithm is incredibly fast because its underlying mechanics are exceedingly simple. It operates through a repetitive loop of assignments and updates:

1.  **Initialization:** First, initialize the $k$ centroids randomly. The most common approach is to randomly pick $k$ distinct instances from the dataset and place the initial centroids at those exact locations.
2.  **Label the instances (Assign):** Assign each instance in the dataset to the centroid that is closest to it (this creates the Voronoi boundaries).
3.  **Update the centroids:** Calculate the mean (average) position of all instances assigned to each centroid, and move the centroid to that new mean location.
4.  **Repeat:** Repeat steps 2 and 3 until the centroids stop moving (this is called *convergence*).

*(Note: In the code, we set `init="random"` and `n_init=1` purely for educational purposes to see the raw algorithm in action. Scikit-Learn's defaults are actually much smarter!)*

### 5. K-Means Variability and Sub-optimal Solutions

Because the original K-Means algorithm relies entirely on a random initialization, it involves a degree of luck. 

If you are unlucky with your random seed, the algorithm might converge to a **sub-optimal solution** (a local optimum). As shown in the visualizations, different random seeds can lead to completely different—and sometimes highly inaccurate—clusterings where multiple clusters are merged or split incorrectly.

```python
# If you happen to know good starting locations, you can pass them manually!
good_init = np.array([[-3, 3], [-3, 2], [-3, 1], [-1, 2], [0, 2]])
kmeans = KMeans(n_clusters=5, init=good_init, n_init=1, random_state=42)
kmeans.fit(X)
```
However, in real-world unsupervised learning, we almost never know where the "good" initial locations are!

### 6. Evaluating K-Means: Inertia

Since clustering is an unsupervised task, we don't have target labels to calculate standard accuracy. Instead, we evaluate a K-Means model by measuring the distance between each instance and its assigned centroid. 

This metric is called **Inertia** (also known as distortion). It is the sum of the squared distances between each training instance and its closest centroid.
*   **Lower Inertia = Better Model** (It means the clusters are dense and tightly packed).

```python
# Check the inertia of a trained model
kmeans.inertia_
# Output: 211.59...

# Scikit-Learn's score() method returns negative inertia 
# to comply with the rule that "greater is better" for scores.
kmeans.score(X)
# Output: -211.59...
```

### 7. Multiple Initializations (`n_init`)

To solve the "K-Means Variability" problem (getting stuck in sub-optimal local minima due to a bad random start), Scikit-Learn uses a simple but effective trick: it runs the algorithm multiple times with different random initializations and keeps the best model.

The number of times it runs is controlled by the `n_init` hyperparameter.
*   When you call `fit()`, Scikit-Learn runs the K-Means algorithm `n_init` times behind the scenes.
*   It calculates the **Inertia** for each run.
*   It returns the model that achieved the **lowest Inertia**.

### 8. The Power of `n_init` in Action

By setting `n_init=10` (which is historically the default behavior in Scikit-Learn when using random initialization), we allow the algorithm to run completely from scratch 10 times with different random seeds. 

```python
kmeans_rnd_10_inits = KMeans(n_clusters=5, init="random", n_init=10, random_state=2)
kmeans_rnd_10_inits.fit(X)
```
As visualized in the resulting decision boundaries, this approach effectively solves the variability problem. The algorithm evaluates the Inertia for all 10 runs, discards the sub-optimal solutions (where centroids got stuck in local minima), and seamlessly returns the single best model with a flawless Voronoi tessellation (assuming we guessed the correct $k$).

### 9. Smart Initialization: K-Means++

Random initialization can lead to sub-optimal solutions. A brilliant improvement called **K-Means++** (introduced in 2006) solves this by selecting initial centroids that are naturally far away from one another. 

**How it works:**
1. Pick one centroid randomly from the dataset.
2. Pick the next centroid by giving a higher probability to instances that are *further away* from the already chosen centroids.
3. Repeat until $k$ centroids are chosen.

*Note: Scikit-Learn's `KMeans` class uses K-Means++ by default (`init="k-means++"`). Because it is so reliable, `n_init` defaults to 1 when using this initialization.*

### 10. Speeding up K-Means

*   **Accelerated K-Means:** Scikit-Learn accelerates the algorithm by avoiding many unnecessary distance calculations. It exploits the *triangle inequality* theorem to keep track of lower and upper bounds for distances between instances and centroids (Elkan's algorithm).
*   **Mini-Batch K-Means:** Instead of using the full dataset at each iteration, you can use mini-batches. This speeds up the algorithm typically by a factor of 3 to 4. It is extremely useful for massive datasets that do not fit in memory, especially when combined with the `np.memmap` class!

```python
from sklearn.cluster import MiniBatchKMeans
# Perfect for out-of-core learning!
minibatch_kmeans = MiniBatchKMeans(n_clusters=5, batch_size=10, random_state=42)
minibatch_kmeans.fit(X_memmap)
```

### 11. Finding the Optimal Number of Clusters ($k$)

If you don't know the optimal number of clusters based on a downstream business need (like choosing $k=3$ for S, M, L shirt sizes), you must evaluate different values of $k$. 
You **cannot** simply choose the $k$ that minimizes Inertia, because Inertia naturally drops toward 0 as $k$ approaches the number of instances.

**Method 1: The Elbow Rule**
Plot the Inertia as a function of $k$. The curve usually resembles an arm. The point where the curve bends sharply—the "elbow"—is generally a good choice for $k$. It represents the point where adding more clusters no longer gives you a significant drop in Inertia.

**Method 2: The Silhouette Score**
A more precise approach is the Silhouette score. It calculates how well each instance sits inside its own cluster compared to other clusters.
The score for a single instance is: `(b - a) / max(a, b)`
*   **$a$**: Mean distance to other instances in the *same* cluster.
*   **$b$**: Mean distance to instances in the *next closest* cluster.

The overall Silhouette Score is the mean silhouette coefficient over all instances. It ranges from **-1 to +1**.
*   **Close to +1:** The instance is perfectly inside its own cluster and far from others.
*   **Close to 0:** The instance is right on the cluster boundary.
*   **Close to -1:** The instance might have been assigned to the wrong cluster.

```python
from sklearn.metrics import silhouette_score
score = silhouette_score(X, kmeans.labels_)
```

### 12. Deep Dive: Silhouette Diagrams

While the mean Silhouette score is helpful, plotting a **Silhouette Diagram** provides a much richer visualization. It plots the silhouette coefficient of every single instance, sorted by cluster.

*   **Height/Thickness:** Represents the size of the cluster (number of instances).
*   **Width (x-axis):** Represents the silhouette coefficient. We want instances to pass the red dashed line (the mean score).

**Why $k=5$ might beat $k=4$:** 
Even if $k=4$ has a slightly higher overall mean silhouette score, its diagram might reveal imbalanced cluster sizes. In contrast, $k=5$ often shows clusters of roughly similar sizes, with all clusters safely crossing the mean dashed line. In many real-world scenarios, clusters of similar sizes are preferable, making $k=5$ the superior choice.

### 13. The Limits of K-Means

K-Means is incredibly fast and scalable, but it has a major structural flaw: **it assumes clusters are spherical and have similar sizes.**

Because K-Means only cares about the distance to the centroids, it struggles heavily with elongated shapes (elliptical blobs) or clusters with varying densities. 
*   **The Inertia Trap:** When dealing with non-spherical clusters, Inertia is no longer a reliable metric. As seen in experiments, a visibly "bad" clustering model that cuts a long cluster in half might actually have a *lower* Inertia than a conceptually "good" model. 
*   **Solution:** For datasets with complex, elliptical, or varying-density clusters, K-Means is not the right tool. You should use algorithms like **Gaussian Mixture Models (GMMs)** instead.

### 14. Real-World Application: Image Segmentation (Color Quantization)

Clustering can be used to segment an image or compress its color palette. In this application, we treat every single pixel as an individual data instance (a point in a 3D color space).

**The Process:**
1.  **Flatten the Image:** An image loaded with shape `(height, width, 3)` is reshaped to a 2D array of shape `(height * width, 3)`. We completely discard the spatial coordinates (x, y) and only keep the RGB color channels as our features ($X$).
2.  **Cluster the Colors:** We run K-Means on this massive list of pixels (e.g., setting `n_clusters=8`). The algorithm groups similar colors together and finds 8 "average" colors (the centroids).
3.  **Replace Colors:** Every pixel's original color is replaced with the color of the centroid it was assigned to.
4.  **Reconstruct:** We reshape the flat list back into the original image dimensions `(height, width, 3)`. 

```python
import numpy as np
from sklearn.cluster import KMeans

# 1. Reshape the image (discard coordinates, keep RGB)
X = image.reshape(-1, 3) 

# 2. Find the top 8 colors in the image
kmeans = KMeans(n_clusters=8, random_state=42).fit(X)

# 3. Replace each pixel with its cluster's centroid color
segmented_img = kmeans.cluster_centers_[kmeans.labels_]

# 4. Reshape back to the original image dimensions to view it
segmented_img = segmented_img.reshape(image.shape)
```
*Notice in the visualizations how the image loses fine details as $k$ decreases, ultimately reducing the entire scene to just 2 dominant colors (e.g., dark green and yellow).*

### 15. Semi-Supervised Learning with Clustering

When we have plenty of unlabeled instances but very few labeled ones, clustering can drastically improve our supervised model's performance.

#### Step 1: Find Representative Images
Instead of randomly choosing instances to label manually, we cluster the entire training set first. Then, we find the instance closest to each centroid. These are our "representative" images. Labeling these specific images provides a much higher accuracy than labeling random ones because they represent the core of each cluster and avoid confusing outliers.

```python
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
import numpy as np

k = 50
kmeans = KMeans(n_clusters=k, random_state=42)
# fit_transform returns the distance of each instance to all 50 centroids
X_digits_dist = kmeans.fit_transform(X_train)

# argmin(axis=0) finds the index of the closest instance to each centroid
representative_digit_idx = X_digits_dist.argmin(axis=0)
X_representative_digits = X_train[representative_digit_idx]

# Imagine we now manually label these 50 specific images -> y_representative_digits
```

#### Step 2: Label Propagation
Once we have labeled the 50 representative instances, we propagate (copy) their labels to *all* other instances in their respective clusters. This massively increases our labeled training set without any extra human effort!

```python
y_train_propagated = np.empty(len(X_train), dtype=np.int32)
for i in range(k):
    y_train_propagated[kmeans.labels_ == i] = y_representative_digits[i]
```

#### Step 3: Eliminate Outliers for Better Accuracy
Propagating labels to instances near the cluster boundaries (the outliers) can introduce noise, as these instances are often blurry or ambiguous. We solve this by propagating labels *only* to the instances that are closest to their centroids (e.g., the closest 50%) and ignoring the rest.

```python
percentile_closest = 50
X_cluster_dist = X_digits_dist[np.arange(len(X_train)), kmeans.labels_]

for i in range(k):
    in_cluster = (kmeans.labels_ == i)
    cluster_dist = X_cluster_dist[in_cluster]
    cutoff_distance = np.percentile(cluster_dist, percentile_closest)
    above_cutoff = (X_cluster_dist > cutoff_distance)
    X_cluster_dist[in_cluster & above_cutoff] = -1

# Filter out the outliers (which are now marked as -1)
partially_propagated = (X_cluster_dist != -1)
X_train_partially_propagated = X_train[partially_propagated]
y_train_partially_propagated = y_train_propagated[partially_propagated]

# Train the final supervised model on this high-quality, noise-free dataset!
log_reg = LogisticRegression(max_iter=10_000)
log_reg.fit(X_train_partially_propagated, y_train_partially_propagated)
```

### 16. Active Learning

To continue improving the model iteratively, you can use **Active Learning**. 
Instead of randomly asking human experts for more labels, you focus human effort only on the instances that the classifier is *least sure about* (often the outliers we ignored during propagation). Once these hard examples are manually labeled, you retrain the model with the new data.

### 17. DBSCAN: Density-Based Spatial Clustering

While K-Means assumes clusters are spherical, **DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) groups together instances that are closely packed together (dense regions) and marks instances in low-density regions as anomalies. It can discover clusters of any shape!

#### How DBSCAN Thinks (The Two Parameters)
DBSCAN relies on two crucial hyperparameters:
1.  **`eps` ($\epsilon$):** The radius of the neighborhood. It defines how far the algorithm should look from a single point.
2.  **`min_samples`:** The minimum number of points required within the `eps` radius for a point to be considered a "Core" point.

#### The Three Types of Points
Based on these parameters, DBSCAN categorizes every instance into one of three types:
*   **Core Points:** Instances that have at least `min_samples` within their `eps` neighborhood. They form the dense bulk of the cluster.
*   **Non-core (Border) Points:** Instances that do not have enough neighbors themselves, but fall within the `eps` neighborhood of a Core point.
*   **Anomalies / Noise:** Instances that are neither core nor border points. DBSCAN assigns them a cluster label of `-1`.

```python
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons

# Generate a dataset of two interleaving half-circles (moons)
X, y = make_moons(n_samples=1000, noise=0.05, random_state=42)

# Initialize and fit DBSCAN
dbscan = DBSCAN(eps=0.2, min_samples=5)
dbscan.fit(X)

# View the labels (Notice the -1 labels which represent anomalies!)
dbscan.labels_[:10]

# Access the indices of the Core points
dbscan.core_sample_indices_
```

#### Visualizing the Moons (Tuning `eps`)
If you look at the visualization of the `make_moons` dataset:
*   When `eps=0.05` (too small), the algorithm is too strict. It breaks the moons into many tiny fragmented clusters and flags many points as anomalies (red crosses).
*   When `eps=0.20` (just right), it perfectly captures the continuous, non-spherical shape of the two moons!

#### The Prediction Trick (Using KNN)
**Important Note:** Unlike K-Means, the `DBSCAN` class in Scikit-Learn does *not* have a `predict()` method for new, unseen instances. It only clusters the data it was trained on. 
To predict the cluster of a new instance, a common trick is to train a Supervised Learning classifier (like **K-Nearest Neighbors**) using only the *Core points* identified by DBSCAN!

```python
from sklearn.neighbors import KNeighborsClassifier
import numpy as np

# Train a KNN classifier exclusively on DBSCAN's Core points
knn = KNeighborsClassifier(n_neighbors=50)
knn.fit(dbscan.components_, dbscan.labels_[dbscan.core_sample_indices_])

# Now we can predict the cluster of brand new data points!
X_new = np.array([[-0.5, 0], [0, 0.5], [1, -0.1], [2, 1]])
knn.predict(X_new)
```

### 18. Spectral Clustering

Unlike K-Means (which looks for spherical centers) or DBSCAN (which looks for density), **Spectral Clustering** treats the data like a complex network or a graph.

#### How it works:
1.  **The Network (Affinity Matrix):** It measures the distance between all instances and creates a network of connections. You can think of this as tying "ropes" between data points based on how close they are.
2.  **Cutting the Ties:** It looks for the weakest connections in this massive network and makes mathematical cuts to separate the graph into distinct clusters.

#### The `gamma` Parameter (The Length of the Rope)
The most critical parameter is `gamma`, which determines how strict the algorithm is about connecting points:
*   **High `gamma` (e.g., 100):** Short ropes. Only points that are very close to each other are connected. This is perfect for complex, intertwined shapes (like the moons dataset), as it stops the algorithm from accidentally connecting points across empty spaces.
*   **Low `gamma` (e.g., 1):** Long ropes. Points that are further away can still be connected. In non-spherical datasets, a low gamma might accidentally bridge the gap between two distinct clusters and ruin the grouping.

```python
from sklearn.cluster import SpectralClustering

# Using a high gamma (short ropes) for the complex moons dataset
sc = SpectralClustering(n_clusters=2, gamma=100, random_state=42)
sc.fit(X)

# The algorithm calculates the network of similarities internally (0 to 1)
print(sc.affinity_matrix_.round(2))
```

### 19. Agglomerative Clustering

While K-Means tries to break the dataset into clusters from the top down, **Agglomerative Clustering** uses a **Bottom-Up** approach. 

#### How it works:
1.  **Initialization:** It starts by treating every single data point as its own individual cluster (e.g., 100 points = 100 clusters).
2.  **Merging:** At each step, it finds the two clusters that are closest to each other and merges them into a single cluster.
3.  **Completion:** It repeats this merging process until all points are merged into one giant cluster (or until it reaches a specific number of clusters if you define it).

#### The `linkage` Parameter
When merging two clusters, how do we define the "distance" between them? The `linkage` parameter decides this:
*   `complete`: Computes the maximum distance between points in the two clusters.
*   `average`: Computes the average distance between points in the two clusters.
*   `single`: Computes the minimum distance between points in the two clusters.

```python
from sklearn.cluster import AgglomerativeClustering
import numpy as np

# A simple 1D dataset: points at 0, 2, 5, and 8.5
X = np.array([0, 2, 5, 8.5]).reshape(-1, 1)

# Using complete linkage to cluster the points
agg = AgglomerativeClustering(linkage="complete").fit(X)

# The children_ attribute shows the history of the merges!
# Row 1: [0, 1] means point 0 and point 1 were merged first (they form group 4)
# Row 2: [2, 3] means point 2 and point 3 were merged next (they form group 5)
# Row 3: [4, 5] means group 4 and group 5 were merged at the end
print(agg.children_)
```

### 20. Gaussian Mixture Models (GMM)

While K-Means strictly forces clusters to be spherical, a **Gaussian Mixture Model** assumes that the dataset is generated from a mixture of several Gaussian (Normal) distributions. This allows GMMs to form flexible, ellipsoidal cluster boundaries of any size, shape, and orientation.

#### The Engine: EM Algorithm
GMM uses the **Expectation-Maximization (EM)** algorithm to estimate three crucial parameters for each cluster:
1.  `means_`: The center of the ellipsoid.
2.  `covariances_`: The shape, size, and orientation of the ellipsoid.
3.  `weights_`: The relative size of the cluster (what percentage of total data belongs to it).

```python
from sklearn.mixture import GaussianMixture

# n_init=10 ensures the EM algorithm runs 10 times with different initializations 
# and keeps the best model to avoid bad local optima.
gm = GaussianMixture(n_components=3, n_init=10, random_state=42)
gm.fit(X)
```

#### Hard vs. Soft Clustering
GMM gives you the option to force a final decision or ask for the model's level of certainty:
*   **Hard Clustering (`predict`):** Assigns each instance to a single cluster.
*   **Soft Clustering (`predict_proba`):** Returns the probability that a given instance belongs to each of the clusters. Perfect for points overlapping the boundaries!

```python
# Returns exact cluster labels [0, 1, or 2]
gm.predict(X)

# Returns probabilities, e.g., [0.01, 0.97, 0.02]
gm.predict_proba(X).round(3)
```

#### A Generative Model
Because GMM learns the actual mathematical distribution of the data, it is a **Generative Model**. You can ask it to generate brand-new, realistic data points that look just like your original training data.

```python
# Generate 6 completely new instances based on the learned patterns
X_new, y_new = gm.sample(6)
```

#### Anomaly Detection with GMM
The `score_samples()` method estimates the log of the probability density function (PDF) at any location. Instances located in dense regions receive high scores, while instances in empty, sparse regions receive highly negative scores. We can isolate these low-scoring instances to detect anomalies (e.g., defective products, fraud).

```python
import numpy as np

# Calculate the density score for all instances
densities = gm.score_samples(X)

# Define a threshold (e.g., the 4% of data with the lowest density scores)
density_threshold = np.percentile(densities, 4)

# Filter the dataset to find the anomalies
anomalies = X[densities < density_threshold]
```

#### Controlling the Shape: `covariance_type`
You can force the algorithm to restrict the shape of the clusters using the `covariance_type` hyperparameter:
*   `"full"` (default): No constraints. Ellipsoids can take any shape, size, and angle.
*   `"tied"`: All clusters must share the exact same ellipsoid shape, size, and angle.
*   `"spherical"`: All clusters must be perfectly spherical (but can have different diameters).
*   `"diag"`: Ellipsoids can be any size, but their axes must be parallel to the coordinate axes (no tilted ellipsoids).

### 21. Anomaly Detection with Gaussian Mixtures

Gaussian Mixtures are highly effective for **Anomaly Detection**. Instances located in low-density regions (far from the centers of the ellipsoids) can be considered anomalies or outliers.

#### How to isolate anomalies:
1.  Use `score_samples()` to get the density score of every instance.
2.  Define a threshold based on the expected contamination rate (e.g., if you expect 2% of your data to be anomalies, find the density value at the 2nd percentile).
3.  Filter out the instances that fall below this density threshold.

```python
import numpy as np

# 1. Get the log of the probability density function (PDF) for each instance
densities = gm.score_samples(X)

# 2. Define the threshold (e.g., isolating the bottom 2% of scores)
density_threshold = np.percentile(densities, 2)

# 3. Filter the instances that have a score strictly lower than the threshold
anomalies = X[densities < density_threshold]
```

### 22. Selecting the Number of Clusters in GMM (BIC & AIC)

We **cannot** use the Inertia or Silhouette score to find the optimal number of clusters ($k$) in a Gaussian Mixture Model because those metrics assume clusters are perfectly spherical. 

Instead, we find the model that minimizes a theoretical information criterion:
*   **BIC** (Bayesian Information Criterion): $BIC = \log(m)p - 2\log(\hat{L})$
*   **AIC** (Akaike Information Criterion): $AIC = 2p - 2\log(\hat{L})$

#### The Penalty and Reward System
Both formulas work on a balance of penalty and reward:
1.  **Penalty for Complexity ($p$):** $p$ is the number of parameters. Models with too many clusters have high $p$ values and are heavily penalized.
2.  **Reward for Goodness of Fit ($\hat{L}$):** $\hat{L}$ is the maximized likelihood function. It measures how perfectly the ellipsoids fit the data. Good fits reduce the final BIC/AIC score.

**Goal:** We want to find the $k$ that results in the **lowest possible (minimum)** BIC or AIC score.

#### Implementation in Scikit-Learn
You can easily get the BIC and AIC scores from a trained model:

```python
# Assuming 'gm' is your trained GaussianMixture model
print("BIC:", gm.bic(X))
print("AIC:", gm.aic(X))
```

#### Finding the Optimal $k$ (The Plotting Method)
Just like the Elbow method in K-Means, we train multiple models with different values of $k$ and plot their BIC/AIC scores to find the minimum point.

```python
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture

# Train models with k from 1 to 10
gms_per_k = [GaussianMixture(n_components=k, n_init=10, random_state=42).fit(X) for k in range(1, 11)]

# Extract BIC and AIC scores
bics = [model.bic(X) for model in gms_per_k]
aics = [model.aic(X) for model in gms_per_k]

# Plotting the scores
plt.plot(range(1, 11), bics, "bo-", label="BIC")
plt.plot(range(1, 11), aics, "go--", label="AIC")
plt.xlabel("$k$ (Number of Clusters)")
plt.ylabel("Information Criterion")
plt.legend()
plt.show()

# The optimal k is where the plot reaches its absolute minimum!
```

### 23. Bayesian Gaussian Mixture Models

Finding the optimal number of clusters manually (using BIC/AIC plots) can be tedious. The **`BayesianGaussianMixture`** class automates this process.

#### How it works:
Instead of searching for the exact $k$, you set `n_components` to a value that you have good reason to believe is *greater* than the optimal number of clusters (e.g., 10). The algorithm will automatically detect the unnecessary clusters and give them a weight equal (or very close) to zero, effectively eliminating them.

```python
from sklearn.mixture import BayesianGaussianMixture

# We guess 10, but let the model decide how many it actually needs
bgm = BayesianGaussianMixture(n_components=10, n_init=10, max_iter=500, random_state=42)
bgm.fit(X)

# Check the weights of the clusters
print(bgm.weights_.round(2))
# Output example: [0.4, 0.21, 0.39, 0., 0., 0., 0., 0., 0., 0.]
# The model successfully identified that only 3 components are needed!
```

#### Limitation of GMMs
Even advanced models like Bayesian GMMs have a fundamental limitation: they inherently assume clusters are ellipsoidal. If you apply them to complex, non-ellipsoidal shapes (like the moons dataset), they will attempt to approximate the shape using multiple smaller ellipsoids instead of detecting the true underlying geometric shape. While this is bad for clustering, the resulting density map can still be useful for Anomaly Detection.

### Chapter 8: Unsupervised Learning – Core Concepts & Review

#### Part 1: Fundamentals of Clustering
1. **Definition & Examples:** Clustering is the unsupervised task of grouping similar instances together. Popular algorithms include K-Means, DBSCAN, Agglomerative Clustering, Spectral Clustering, BIRCH, and Mean-Shift.
2. **Main Applications:** It is heavily used in customer segmentation, data analysis, recommender systems, image segmentation, semi-supervised learning, and anomaly/novelty detection.
3. **Finding *k* in K-Means:** We cannot just guess the number of clusters. We use two visual techniques:
   * **The Elbow Rule:** Plotting Inertia (distance to centroids) against *k* and finding the inflection point.
   * **Silhouette Score:** Plotting the mean silhouette coefficient to find the peak, or analyzing silhouette diagrams for cluster balance.

#### Part 2: Semi-Supervised & Active Learning
4. **Label Propagation:** When labeling data is expensive, you can label just a few instances and propagate (copy) those labels to similar unlabeled instances. 
   * *Implementation:* Run K-Means on the whole dataset. Find the representative instance closest to each centroid, manually label it, and propagate that label to the rest of the instances in that cluster.
6. **Active Learning:** Instead of randomly choosing which instances a human expert should label, the algorithm specifically requests labels for the instances it is most confused about (Uncertainty Sampling).

#### Part 3: Algorithm Selection
5. **Scaling vs. Density:** 
   * *Massive Datasets:* K-Means and BIRCH scale very well.
   * *High-Density/Irregular Shapes:* DBSCAN and Mean-Shift are designed to follow dense regions.
7. **Anomaly vs. Novelty Detection:** 
   * *Anomaly Detection:* The training set already contains outliers, and the model learns to isolate them (e.g., catching credit card fraud in historical data).
   * *Novelty Detection:* The training set is 100% "clean" and normal. The model only looks for strange items in *new*, unseen data.

#### Part 4: Gaussian Mixtures (GMM)
8. **What is a GMM?** A probabilistic model assuming data is generated from a mixture of Gaussian distributions. It groups data into ellipsoids of varying sizes and angles. It is used for density estimation, clustering, and anomaly detection.
9. **Finding *k* in GMM:** 
   * *Manual:* Plot the BIC or AIC metrics and find the minimum point (Inertia does not work for ellipsoids).
   * *Automatic:* Use `BayesianGaussianMixture`, set a high `n_components`, and let it assign weights of zero to unnecessary clusters.

## Chapter 8 Summary: Unsupervised Learning & Clustering

### 1. K-Means Clustering
*   **Core Logic:** Finds cluster centers (centroids) and updates them iteratively. It strictly assumes clusters are spherical and have similar sizes.
*   **Finding $k$:** Use the **Elbow Method** (plotting Inertia) or the **Silhouette Score** (measuring cluster density and separation).
*   **Use Cases:** Fast and scales well to massive datasets. Highly effective for dimensionality reduction and semi-supervised learning (Label Propagation).

### 2. Advanced Clustering Algorithms
*   **DBSCAN:** Density-based. Groups closely packed points and flags isolated points in empty spaces as anomalies (label `-1`). Perfect for arbitrary, non-spherical shapes. *Note: Uses a separate KNN model to predict labels for new instances.*
*   **Spectral Clustering:** Creates a network (graph) of connections between instances and cuts the weakest links. Strongly relies on the `gamma` parameter to define how close points must be to connect.
*   **Agglomerative Clustering:** A bottom-up hierarchical approach. Starts with individual points as standalone clusters and merges the closest ones step-by-step.

### 3. Gaussian Mixture Models (GMM)
*   **Core Logic:** A probabilistic model using the Expectation-Maximization (EM) algorithm to fit ellipsoids (normal distributions) to the data. It learns `means_`, `covariances_`, and `weights_`.
*   **Key Capabilities:**
    *   **Soft Clustering:** Returns the exact probability of a point belonging to each cluster (`predict_proba`).
    *   **Generative:** Can mathematically generate brand new, realistic data points (`sample()`).
    *   **Anomaly Detection:** Identifies outliers in low-density regions using `score_samples()`.
*   **Finding $k$:** Inertia and Silhouette scores fail here. Instead, we minimize theoretical information criteria (**BIC** or **AIC**) which heavily penalize overly complex models (Occam's Razor).
*   **Bayesian GMM:** Automates the search for $k$. You provide a high maximum number of components, and the algorithm zeroes out the weights of any unnecessary clusters.

### 4. Crucial ML Concepts
*   **Label Propagation:** A cost-saving technique where you run clustering on a massive unlabeled dataset, manually label only the most representative instances (the centroids), and copy those labels to the rest of the cluster.
*   **Active Learning:** Instead of picking random data to label, the algorithm actively asks human experts to label specific instances it is most confused about (Uncertainty Sampling).
*   **Anomaly vs. Novelty Detection:** Anomaly detection finds outliers within a "dirty" training set. Novelty detection trains on a strictly "clean" dataset and looks for strange patterns only in new, unseen data.